## Reading in ITU All Country Data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# importing Data

itu_df = pd.read_csv("../data/raw/itu_extract_all_countries_data.csv")

print(itu_df.head())

  SeriesID SeriesCode                                             Series  \
0      260        i61                                         Population   
1      430      i271G  Percentage of the population covered by at lea...   
2      242      i4214          International bandwidth usage, in Mbit/s    
3      194    i271pop  Percentage of the population covered by a mobi...   
4      260        i61                                         Population   

               Unit                                         Database    Year  \
0             Value  ITU World Telecommunication Indicators Database  2025.0   
1  % of individuals  ITU World Telecommunication Indicators Database  2025.0   
2            Mbit/s  ITU World Telecommunication Indicators Database  2025.0   
3  % of individuals  ITU World Telecommunication Indicators Database  2025.0   
4             Value  ITU World Telecommunication Indicators Database  2025.0   

       AreaName AreaCode DisaggregationGroup Disaggregation1 D

C:\Users\arian\AppData\Local\Temp\ipykernel_38560\1238937381.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  itu_df = pd.read_csv("../data/raw/itu_extract_all_countries_data.csv")


In [4]:
itu_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127425 entries, 0 to 127424
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   SeriesID             127425 non-null  object 
 1   SeriesCode           127425 non-null  object 
 2   Series               127423 non-null  object 
 3   Unit                 127423 non-null  object 
 4   Database             127423 non-null  object 
 5   Year                 127423 non-null  float64
 6   AreaName             127423 non-null  object 
 7   AreaCode             127423 non-null  object 
 8   DisaggregationGroup  79829 non-null   object 
 9   Disaggregation1      78689 non-null   object 
 10  Disaggregation2      31670 non-null   object 
 11  Value                126145 non-null  float64
 12  Notes                20987 non-null   object 
 13  Source               109764 non-null  object 
 14  LastModifiedOn       127421 non-null  object 
dtypes: float64(2), ob

In [5]:
itu_df.shape

(127425, 15)

In [6]:
itu_df.columns

Index(['SeriesID', 'SeriesCode', 'Series', 'Unit', 'Database', 'Year',
       'AreaName', 'AreaCode', 'DisaggregationGroup', 'Disaggregation1',
       'Disaggregation2', 'Value', 'Notes', 'Source', 'LastModifiedOn'],
      dtype='object')

In [7]:
# Identifying all unique indicators and their codes

indicators = (
    itu_df[["SeriesCode", "Series"]]
    .drop_duplicates()
    .sort_values("Series")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(indicators)

,SeriesCode,Series
0,i271mw,Active mobile-broadband subscriptions
1,i911mw,Active mobile-broadband subscriptions per 100 inhabitants
2,i271mwa,Active subscriptions to LTE/WiMAX mobile-broadband networks
3,i4213cab,Cable modem Internet subscriptions
4,i4213dsl,DSL Internet subscriptions
5,i271mb_active,Data and voice mobile-broadband subscriptions
6,i271md,Data-only mobile-broadband subscriptions
7,i911md,Dedicated mobile data subscriptions per 100 inhabitants
8,xHH6_IDI,Estimated proportion of households with Internet access at home
9,i4213ftthb,Fibre-to-the-home/building Internet subscriptions


In [8]:
# Checking the years available in the dataset

itu_df["Year"].unique()

array([2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018., 2017.,
       2016., 2015., 2014., 2013., 2012., 2011., 2010., 2009., 2008.,
         nan, 2007., 2006., 2005., 2004., 2003., 2002., 2001., 2000.])

## Importing Fixed Broadband Subscriptions

In [22]:
# importing Fixed Broadband subscriptions (per 100 inhabitants) data

fixed_bb = pd.read_csv("../data/raw/fixed_broadband_per_100.csv")

fixed_bb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4805 entries, 0 to 4804
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SeriesID             4805 non-null   int64  
 1   SeriesCode           4805 non-null   object 
 2   Series               4805 non-null   object 
 3   Unit                 4805 non-null   object 
 4   Database             4805 non-null   object 
 5   Year                 4805 non-null   int64  
 6   AreaName             4805 non-null   object 
 7   AreaCode             4805 non-null   object 
 8   DisaggregationGroup  234 non-null    object 
 9   Disaggregation1      0 non-null      float64
 10  Disaggregation2      0 non-null      float64
 11  Value                4805 non-null   float64
 12  Notes                164 non-null    object 
 13  Source               704 non-null    object 
 14  LastModifiedOn       4805 non-null   object 
dtypes: float64(3), int64(2), object(10)
me

In [23]:
fixed_bb.shape

(4805, 15)

In [27]:
fixed_bb[["Series","SeriesCode","Unit"]].drop_duplicates()

,Series,SeriesCode,Unit
0,Fixed broadband subscriptions per 100 inhabitants,i992b,per 100 people


In [28]:
# Check year range and country coverage

print("Earliest year:", fixed_bb["Year"].min())
print("Latest year:", fixed_bb["Year"].max())

print("Unique countries/areas:", fixed_bb["AreaCode"].nunique())

print("Duplicate country-year rows:",
      fixed_bb.duplicated(subset=["AreaCode", "Year"]).sum())

print("Missing subscription values:",
      fixed_bb["Value"].isna().sum())

Earliest year: 2000
Latest year: 2025
Unique countries/areas: 231
Duplicate country-year rows: 5
Missing subscription values: 0


In [29]:
# Display all duplicated country-year observations

duplicates = fixed_bb[
    fixed_bb.duplicated(
        subset=["AreaCode", "Year"],
        keep=False
    )
].sort_values(["AreaCode", "Year"])

display(
    duplicates[
        ["AreaName", "AreaCode", "Year",
         "SeriesCode", "Value",
         "DisaggregationGroup", "Disaggregation1"]
    ]
)

,AreaName,AreaCode,Year,SeriesCode,Value,DisaggregationGroup,Disaggregation1
12,Burundi,BDI,2025,i992b,0.045219,NaN,NaN
13,Burundi,BDI,2025,i992b,0.045219,NaN,NaN
136,Georgia,GEO,2025,i992b,31.761873,NaN,NaN
137,Georgia,GEO,2025,i992b,31.761873,NaN,NaN
49,Honduras,HND,2025,i992b,5.072766,NaN,NaN
50,Honduras,HND,2025,i992b,5.072766,NaN,NaN
73,Mauritius,MUS,2025,i992b,28.069511,NaN,NaN
74,Mauritius,MUS,2025,i992b,28.069511,NaN,NaN
308,South Sudan,SSD,2024,i992b,0.001675,NaN,NaN
309,South Sudan,SSD,2024,i992b,0.001675,NaN,NaN


In [30]:
# Remove duplicate country-year observations

fixed_bb_clean = fixed_bb.drop_duplicates(
    subset=["AreaCode", "Year"],
    keep="first"
).copy()

print("Original rows:", len(fixed_bb))
print("Cleaned rows:", len(fixed_bb_clean))
print("Remaining duplicates:",
      fixed_bb_clean.duplicated(
          subset=["AreaCode", "Year"]
      ).sum())

Original rows: 4805
Cleaned rows: 4800
Remaining duplicates: 0


In [31]:
# Select and rename the necessary columns

fixed_bb_clean = fixed_bb_clean[
    ["AreaCode", "Year", "Value"]
].rename(
    columns={"Value": "Fixed_Broadband_per_100"}
)

display(fixed_bb_clean.head())

,AreaCode,Year,Fixed_Broadband_per_100
0,ZAF,2025,5.433783
1,AGO,2025,0.377889
2,ALB,2025,26.450726
3,DZA,2025,14.512423
4,AND,2025,53.247129


In [32]:
from pathlib import Path

# Create processed folder if needed
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Save cleaned fixed-broadband data
fixed_bb_clean.to_csv(
    "../data/processed/fixed_bb_clean.csv",
    index=False
)

print("Fixed broadband data saved!")

Fixed broadband data saved!


## Displaying all ITU indicators

In [40]:
# Display all indicators available in the ITU dataset

all_indicators = (
    itu_df[["SeriesCode", "Series", "Unit"]]
    .dropna(subset=["Series"])
    .drop_duplicates()
    .sort_values("Series")
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows",None,
    "display.max_colwidth", None
):
    display(all_indicators)


print("Total unique indicator codes:",
      all_indicators["SeriesCode"].nunique())

,SeriesCode,Series,Unit
0,i271mw,Active mobile-broadband subscriptions,Value
1,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,per 100 people
2,i271mwa,Active subscriptions to LTE/WiMAX mobile-broadband networks,Value
3,i4213cab,Cable modem Internet subscriptions,Value
4,i4213dsl,DSL Internet subscriptions,Value
5,i271mb_active,Data and voice mobile-broadband subscriptions,Value
6,i271md,Data-only mobile-broadband subscriptions,Value
7,i911md,Dedicated mobile data subscriptions per 100 inhabitants,per 100 people
8,xHH6_IDI,Estimated proportion of households with Internet access at home,% of households
9,i4213ftthb,Fibre-to-the-home/building Internet subscriptions,Value


Total unique indicator codes: 200


In [41]:
# Summarize data availability for every indicator

indicator_summary = (
    itu_df.groupby(
        ["SeriesCode", "Series", "Unit"],
        dropna=False
    )
    .agg(
        total_rows=("Value", "size"),
        available_values=("Value", "count"),
        countries_areas=("AreaCode", "nunique"),
        first_year=("Year", "min"),
        last_year=("Year", "max")
    )
    .reset_index()
)

indicator_summary["missing_values"] = (
    indicator_summary["total_rows"]
    - indicator_summary["available_values"]
)

display(indicator_summary.sort_values("Series"))

,SeriesCode,Series,Unit,total_rows,available_values,countries_areas,first_year,last_year,missing_values
183,i271mw,Active mobile-broadband subscriptions,Value,3134,3131,195,2002.0,2025.0,3
196,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,per 100 people,3357,3357,209,2002.0,2025.0,0
184,i271mwa,Active subscriptions to LTE/WiMAX mobile-broadband networks,Value,1182,1054,176,2013.0,2025.0,128
187,i4213cab,Cable modem Internet subscriptions,Value,2975,2829,195,2000.0,2025.0,146
188,i4213dsl,DSL Internet subscriptions,Value,3512,3443,194,2000.0,2025.0,69
181,i271mb_active,Data and voice mobile-broadband subscriptions,Value,1886,1805,182,2007.0,2025.0,81
182,i271md,Data-only mobile-broadband subscriptions,Value,1890,1782,179,2007.0,2025.0,108
195,i911md,Dedicated mobile data subscriptions per 100 inhabitants,per 100 people,1660,1660,161,2007.0,2024.0,0
201,xHH6_IDI,Estimated proportion of households with Internet access at home,% of households,4002,4001,195,2000.0,2025.0,1
189,i4213ftthb,Fibre-to-the-home/building Internet subscriptions,Value,2210,2165,178,2000.0,2025.0,45


In [42]:
# General dataset overview

print("Dataset shape:", itu_df.shape)
print("Columns:", itu_df.columns.tolist())
print("Years:", sorted(itu_df["Year"].dropna().unique()))
print("Countries/areas:", itu_df["AreaCode"].nunique())

print("\nMissing values by column:")
display(itu_df.isna().sum())

Dataset shape: (127425, 15)
Columns: ['SeriesID', 'SeriesCode', 'Series', 'Unit', 'Database', 'Year', 'AreaName', 'AreaCode', 'DisaggregationGroup', 'Disaggregation1', 'Disaggregation2', 'Value', 'Notes', 'Source', 'LastModifiedOn']
Years: [2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0, 2025.0]
Countries/areas: 210

Missing values by column:


SeriesID                    0
SeriesCode                  0
Series                      2
Unit                        2
Database                    2
Year                        2
AreaName                    2
AreaCode                    2
DisaggregationGroup     47596
Disaggregation1         48736
Disaggregation2         95755
Value                    1280
Notes                  106438
Source                  17661
LastModifiedOn              4
dtype: int64

In [43]:
# Search for internet access barriers

barrier_indicators = (
    itu_df[
        itu_df["Series"].str.contains(
            "reason|barrier",
            case=False,
            na=False
        )
    ][["SeriesCode", "Series", "Unit"]]
    .drop_duplicates()
)

display(barrier_indicators)

,SeriesCode,Series,Unit
262,HHA4213bc-%_HHNOINT-URBAN-,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT - URBAN",% of households without Internet access
263,HHA4213bc-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT - RURAL",% of households without Internet access
264,HHA4213bn-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Do not need the Internet - %_HHNOINT - RURAL",% of households without Internet access
265,HHA4213be-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Have access to the Internet elsewhere - %_HHNOINT",% of households without Internet access
266,HHA4213bc-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT",% of households without Internet access
267,HHA4213bs-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT",% of households without Internet access
269,HHA4213ba-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Internet service is not available in the area - %_HHNOINT - RURAL",% of households without Internet access
270,HHA4213bp-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Privacy or security concerns - %_HHNOINT - RURAL",% of households without Internet access
271,HHA4213bh-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Internet service is available but it does not correspond to household needs - %_HHNOINT",% of households without Internet access
273,HHA4213bn-%_HHNOINT-URBAN-,"HH14 - Households not using the Internet, by type of reason: Do not need the Internet - %_HHNOINT - URBAN",% of households without Internet access


In [44]:
# Inspect the different reasons for not having internet

barrier_data = itu_df[
    itu_df["SeriesCode"].isin(
        barrier_indicators["SeriesCode"]
    )
]

display(
    barrier_data[
        ["Series", "DisaggregationGroup", "Disaggregation1",
         "Disaggregation2"]
    ].drop_duplicates()
)

,Series,DisaggregationGroup,Disaggregation1,Disaggregation2
262,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT - URBAN",Location,Urban,NaN
263,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT - RURAL",Location,Rural,NaN
264,"HH14 - Households not using the Internet, by type of reason: Do not need the Internet - %_HHNOINT - RURAL",Location,Rural,NaN
265,"HH14 - Households not using the Internet, by type of reason: Have access to the Internet elsewhere - %_HHNOINT",NaN,NaN,NaN
266,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT",NaN,NaN,NaN
267,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT",NaN,NaN,NaN
269,"HH14 - Households not using the Internet, by type of reason: Internet service is not available in the area - %_HHNOINT - RURAL",Location,Rural,NaN
270,"HH14 - Households not using the Internet, by type of reason: Privacy or security concerns - %_HHNOINT - RURAL",Location,Rural,NaN
271,"HH14 - Households not using the Internet, by type of reason: Internet service is available but it does not correspond to household needs - %_HHNOINT",NaN,NaN,NaN
273,"HH14 - Households not using the Internet, by type of reason: Do not need the Internet - %_HHNOINT - URBAN",Location,Urban,NaN


In [45]:
# Check availability of actual barrier measurements

barrier_valid = barrier_data.dropna(subset=["Value"])

print("Available observations:", len(barrier_valid))
print("Countries/areas:", barrier_valid["AreaCode"].nunique())
print("Years:", sorted(barrier_valid["Year"].dropna().unique()))

Available observations: 13146
Countries/areas: 87
Years: [2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0, 2025.0]


In [46]:
barrier_summary = (
    barrier_valid.groupby(["SeriesCode", "Series"])
    .agg(
        observations=("Value", "size"),
        countries=("AreaCode", "nunique"),
        first_year=("Year", "min"),
        last_year=("Year", "max")
    )
    .reset_index()
    .sort_values("Series")
)

display(barrier_summary.head(15))

print("Total barrier indicators:", len(barrier_summary))

,SeriesCode,Series,observations,countries,first_year,last_year
81,HHA4213bs-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT",313,81,2013.0,2025.0
82,HHA4213bs-%_HHNOINT-CHILDREN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN",188,59,2013.0,2025.0
83,HHA4213bs-%_HHNOINT-CHILDREN-RURAL,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN - RURAL",130,44,2013.0,2025.0
84,HHA4213bs-%_HHNOINT-CHILDREN-URBAN,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN - URBAN",141,44,2013.0,2025.0
85,HHA4213bs-%_HHNOINT-NOCHILDREN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN",222,66,2013.0,2025.0
86,HHA4213bs-%_HHNOINT-NOCHILDREN-RURAL,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN - RURAL",149,44,2013.0,2025.0
87,HHA4213bs-%_HHNOINT-NOCHILDREN-URBAN,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN - URBAN",160,46,2013.0,2025.0
88,HHA4213bs-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - RURAL",220,59,2013.0,2025.0
89,HHA4213bs-%_HHNOINT-URBAN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - URBAN",236,62,2013.0,2025.0
9,HHA4213bc-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT",278,78,2013.0,2025.0


Total barrier indicators: 90


### Barrier data seems to not have stated until 2013

In [47]:
# Extract the underlying barrier reasons

barrier_reasons = (
    barrier_valid["Series"]
    .str.extract(r"type of reason:\s*(.*?)\s*-\s*%_HHNOINT")[0]
    .dropna()
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

print("Number of distinct reasons:", len(barrier_reasons))
print(barrier_reasons.to_string(index=False))

Number of distinct reasons: 10
                                                Cost of service is too high
                                          Cost of the equipment is too high
                                                   Do not need the Internet
                                      Have access to the Internet elsewhere
Internet service is available but it does not correspond to household needs
                              Internet service is not available in the area
                                               Privacy or security concerns
                                                           cultural reasons
                                            no electricity in the household
                                                               other reason


In [48]:
# Inspect the structure of overall household barrier records

barrier_valid[
    barrier_valid["DisaggregationGroup"].isna()
][
    ["SeriesCode", "Series", "Unit",
     "DisaggregationGroup", "Disaggregation1"]
].drop_duplicates().head(12)

,SeriesCode,Series,Unit,DisaggregationGroup,Disaggregation1
265,HHA4213be-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Have access to the Internet elsewhere - %_HHNOINT",% of households without Internet access,NaN,NaN
266,HHA4213bc-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT",% of households without Internet access,NaN,NaN
267,HHA4213bs-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT",% of households without Internet access,NaN,NaN
271,HHA4213bh-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Internet service is available but it does not correspond to household needs - %_HHNOINT",% of households without Internet access,NaN,NaN
284,HHA4213bn-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Do not need the Internet - %_HHNOINT",% of households without Internet access,NaN,NaN
287,HHA4213ba-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Internet service is not available in the area - %_HHNOINT",% of households without Internet access,NaN,NaN
297,HHA4213bo-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: other reason - %_HHNOINT",% of households without Internet access,NaN,NaN
298,HHA4213bp-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Privacy or security concerns - %_HHNOINT",% of households without Internet access,NaN,NaN
301,HHA4213br-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: cultural reasons - %_HHNOINT",% of households without Internet access,NaN,NaN
3324,HHA4213bd-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: no electricity in the household - %_HHNOINT",% of households without Internet access,NaN,NaN


In [49]:
# Select overall household barrier observations long data format

barriers_clean = barrier_valid[
    barrier_valid["DisaggregationGroup"].isna()
    & barrier_valid["Disaggregation1"].isna()
    & barrier_valid["Disaggregation2"].isna()
].copy()

# Extract the reason from the indicator name
barriers_clean["Barrier_Reason"] = (
    barriers_clean["Series"]
    .str.extract(r"type of reason:\s*(.*?)\s*-\s*%_HHNOINT")[0]
    .str.strip()
)

# Keep relevant columns and rename Value
barriers_clean = barriers_clean[
    ["AreaCode", "AreaName", "Year",
     "Barrier_Reason", "Value"]
].rename(columns={"Value": "Barrier_Percent"})

print("Rows:", len(barriers_clean))
print("Countries/areas:", barriers_clean["AreaCode"].nunique())
print("Unique reasons:", barriers_clean["Barrier_Reason"].nunique())
print("Missing extracted reasons:",
      barriers_clean["Barrier_Reason"].isna().sum())

display(barriers_clean.head())

Rows: 2222
Countries/areas: 87
Unique reasons: 10
Missing extracted reasons: 0


,AreaCode,AreaName,Year,Barrier_Reason,Barrier_Percent
265,AZE,Azerbaijan,2025.0,Have access to the Internet elsewhere,11.400215
266,AZE,Azerbaijan,2025.0,Cost of the equipment is too high,10.099914
267,AZE,Azerbaijan,2025.0,Cost of service is too high,13.800190
271,AZE,Azerbaijan,2025.0,Internet service is available but it does not correspond to household needs,0.500013
284,AZE,Azerbaijan,2025.0,Do not need the Internet,42.699983


In [ ]:
# Check duplicate country-year-reason combinations
print("Duplicates:", barriers_clean.duplicated(
    subset=["AreaCode", "Year", "Barrier_Reason"]
).sum())

# Check percentages outside the expected 0–100 range
print("Invalid percentages:", (
    ~barriers_clean["Barrier_Percent"].between(0, 100)
).sum())

# Check missing values
print("\nMissing values:")
print(barriers_clean.isna().sum())

Duplicates: 0
Invalid percentages: 8

Missing values:
AreaCode           0
AreaName           0
Year               0
Barrier_Reason     0
Barrier_Percent    0
dtype: int64


### Investing the 8 out of range barrier percentages

In [ ]:
# Investigate  barrier percentages

invalid_barriers = barriers_clean[
    ~barriers_clean["Barrier_Percent"].between(0, 100)
]

display(invalid_barriers)

print("Invalid records:", len(invalid_barriers))

,AreaCode,AreaName,Year,Barrier_Reason,Barrier_Percent
56679,SAU,Saudi Arabia,2017.0,Have access to the Internet elsewhere,138.888739
65375,BHR,Bahrain,2016.0,Do not need the Internet,123.364486
65556,BRN,Brunei Darussalam,2016.0,Have access to the Internet elsewhere,103.999321
65557,BRN,Brunei Darussalam,2016.0,Cost of service is too high,223.995927
70918,ARE,United Arab Emirates,2016.0,Have access to the Internet elsewhere,708.073470
70924,ARE,United Arab Emirates,2016.0,Cost of service is too high,334.184564
70925,ARE,United Arab Emirates,2016.0,Cost of the equipment is too high,239.885163
70926,ARE,United Arab Emirates,2016.0,Do not need the Internet,797.409828


Invalid records: 8


In [52]:
# Retain only valid barrier percentages (0–100)

barriers_final = barriers_clean[
    barriers_clean["Barrier_Percent"].between(0, 100)
].copy()

print("Original records:", len(barriers_clean))
print("Removed invalid records:", len(barriers_clean) - len(barriers_final))
print("Final records:", len(barriers_final))

Original records: 2222
Removed invalid records: 8
Final records: 2214


In [53]:
print("Remaining duplicates:",
      barriers_final.duplicated(
          subset=["AreaCode", "Year", "Barrier_Reason"]
      ).sum())

print("Invalid percentages remaining:",
      (~barriers_final["Barrier_Percent"].between(0, 100)).sum())

print("Missing values:", barriers_final.isna().sum().sum())

Remaining duplicates: 0
Invalid percentages remaining: 0
Missing values: 0


In [54]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

barriers_final.to_csv(
    "../data/processed/barriers_final.csv",
    index=False
)

### Identying Core Indicators for Experiment B

In [55]:
# Identify the main indicators for Experiment B

core_indicators = itu_df[
    itu_df["Series"].str.contains(
        "Individuals using the Internet|LTE/WiMAX|mobile-broadband",
        case=False,
        na=False
    )
][["SeriesCode", "Series", "Unit"]].drop_duplicates()

display(core_indicators.sort_values("Series"))

,SeriesCode,Series,Unit
6,i271mw,Active mobile-broadband subscriptions,Value
9,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,per 100 people
12,i271mwa,Active subscriptions to LTE/WiMAX mobile-broadband networks,Value
8,i271mb_active,Data and voice mobile-broadband subscriptions,Value
7,i271md,Data-only mobile-broadband subscriptions,Value
10,i271GA,Percentage of the population covered by at least an LTE/WiMAX mobile network.,% of individuals
5527,i911MB_active,Standard mobile-broadband subscriptions per 100 inhabitants,per 100 people


In [56]:
# Find all internet adoption indicators

internet_indicators = (
    itu_df[
        itu_df["Series"].str.contains(
            "using the Internet|Internet users|Internet use",
            case=False,
            na=False
        )
    ][["SeriesCode", "Series", "Unit"]]
    .drop_duplicates()
    .sort_values("Series")
)

display(internet_indicators.head(20))

print("Matching indicators:", len(internet_indicators))

,SeriesCode,Series,Unit
267,HHA4213bs-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT",% of households without Internet access
466,HHA4213bs-%_HHNOINT-CHILDREN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN",% of households without Internet access
416,HHA4213bs-%_HHNOINT-CHILDREN-RURAL,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN - RURAL",% of households without Internet access
455,HHA4213bs-%_HHNOINT-CHILDREN-URBAN,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - CHILDREN - URBAN",% of households without Internet access
424,HHA4213bs-%_HHNOINT-NOCHILDREN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN",% of households without Internet access
460,HHA4213bs-%_HHNOINT-NOCHILDREN-RURAL,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN - RURAL",% of households without Internet access
459,HHA4213bs-%_HHNOINT-NOCHILDREN-URBAN,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - NOCHILDREN - URBAN",% of households without Internet access
275,HHA4213bs-%_HHNOINT-RURAL-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - RURAL",% of households without Internet access
286,HHA4213bs-%_HHNOINT-URBAN-,"HH14 - Households not using the Internet, by type of reason: Cost of service is too high - %_HHNOINT - URBAN",% of households without Internet access
266,HHA4213bc-%_HHNOINT--,"HH14 - Households not using the Internet, by type of reason: Cost of the equipment is too high - %_HHNOINT",% of households without Internet access


Matching indicators: 92


In [57]:
# Search for internet adoption indicators only

internet_adoption = (
    itu_df[
        itu_df["Series"].str.contains(
            "individuals using|percentage of individuals|internet users",
            case=False,
            na=False
        )
        & ~itu_df["Series"].str.contains(
            "households not using|type of reason",
            case=False,
            na=False
        )
    ][["SeriesCode", "Series", "Unit"]]
    .drop_duplicates()
)

display(internet_adoption.head(15))
print("Matching indicators:", len(internet_adoption))

,SeriesCode,Series,Unit
164,i99H,Internet users (%),% of individuals


Matching indicators: 1


In [58]:
# Check availability of our three main ITU indicators

core_codes = ["i99H", "i271GA", "i911mw"]

core_data = itu_df[
    itu_df["SeriesCode"].isin(core_codes)
].copy()

availability = (
    core_data.groupby(["SeriesCode", "Series"])
    .agg(
        observations=("Value", "count"),
        countries=("AreaCode", "nunique"),
        first_year=("Year", "min"),
        last_year=("Year", "max")
    )
    .reset_index()
)

display(availability)

,SeriesCode,Series,observations,countries,first_year,last_year
0,i271GA,Percentage of the population covered by at least an LTE/WiMAX mobile network.,2411,207,2012.0,2025.0
1,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,3357,209,2002.0,2025.0
2,i99H,Internet users (%),5005,210,2000.0,2025.0


In [60]:
# Check for duplicate country-year-indicator records

duplicates = core_data.duplicated(
    subset=["AreaCode", "Year", "SeriesCode"]
).sum()

print("Duplicate records:", duplicates)

Duplicate records: 4


In [61]:
# Identify duplicate country-year-indicator records

core_duplicates = core_data[
    core_data.duplicated(
        subset=["AreaCode", "Year", "SeriesCode"],
        keep=False
    )
].sort_values(["AreaCode", "Year", "SeriesCode"])

display(
    core_duplicates[
        ["AreaName", "AreaCode", "Year",
         "SeriesCode", "Series", "Value",
         "DisaggregationGroup", "Disaggregation1"]
    ]
)

,AreaName,AreaCode,Year,SeriesCode,Series,Value,DisaggregationGroup,Disaggregation1
2113,Honduras,HND,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,50.032392,NaN,NaN
2114,Honduras,HND,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,50.032392,NaN,NaN
2780,Kuwait,KWT,2025.0,i271GA,Percentage of the population covered by at least an LTE/WiMAX mobile network.,100.000000,NaN,NaN
2783,Kuwait,KWT,2025.0,i271GA,Percentage of the population covered by at least an LTE/WiMAX mobile network.,100.000000,NaN,NaN
4352,Sao Tome and Principe,STP,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,46.227326,NaN,NaN
4353,Sao Tome and Principe,STP,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,46.227326,NaN,NaN
5211,Zimbabwe,ZWE,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,74.117397,NaN,NaN
5212,Zimbabwe,ZWE,2025.0,i911mw,Active mobile-broadband subscriptions per 100 inhabitants,74.117397,NaN,NaN


In [62]:
# Remove duplicate country-year-indicator records

core_clean = core_data.drop_duplicates(
    subset=["AreaCode", "Year", "SeriesCode"],
    keep="first"
).copy()

print("Original rows:", len(core_data))
print("Cleaned rows:", len(core_clean))
print("Remaining duplicates:",
      core_clean.duplicated(
          subset=["AreaCode", "Year", "SeriesCode"]
      ).sum())

Original rows: 10781
Cleaned rows: 10777
Remaining duplicates: 0


In [63]:
# Reshape the dataset to one row per country-year

itu_wide = core_clean.pivot(
    index=["AreaCode", "AreaName", "Year"],
    columns="SeriesCode",
    values="Value"
).reset_index()

# Rename columns for easier analysis
itu_wide = itu_wide.rename(columns={
    "i99H": "Internet_Users_Pct",
    "i271GA": "LTE_Coverage_Pct",
    "i911mw": "Mobile_Broadband_per_100"
})

itu_wide.columns.name = None

print("Dataset shape:", itu_wide.shape)
display(itu_wide.head())

Dataset shape: (5234, 6)


,AreaCode,AreaName,Year,LTE_Coverage_Pct,Mobile_Broadband_per_100,Internet_Users_Pct
0,AFG,Afghanistan,2001.0,NaN,NaN,0.004723
1,AFG,Afghanistan,2002.0,NaN,NaN,0.004561
2,AFG,Afghanistan,2003.0,NaN,NaN,0.087891
3,AFG,Afghanistan,2004.0,NaN,NaN,0.105809
4,AFG,Afghanistan,2005.0,NaN,NaN,1.224148


In [64]:
# Convert Year from float to integer
itu_wide["Year"] = itu_wide["Year"].astype(int)

display(itu_wide.head())

,AreaCode,AreaName,Year,LTE_Coverage_Pct,Mobile_Broadband_per_100,Internet_Users_Pct
0,AFG,Afghanistan,2001,NaN,NaN,0.004723
1,AFG,Afghanistan,2002,NaN,NaN,0.004561
2,AFG,Afghanistan,2003,NaN,NaN,0.087891
3,AFG,Afghanistan,2004,NaN,NaN,0.105809
4,AFG,Afghanistan,2005,NaN,NaN,1.224148


In [65]:
# Check data availability after reshaping

print("Countries/areas:",
      itu_wide["AreaCode"].nunique())

print("\nAvailable values by indicator:")
print(
    itu_wide[
        ["Internet_Users_Pct",
         "LTE_Coverage_Pct",
         "Mobile_Broadband_per_100"]
    ].notna().sum()
)

# Count country-years with both coverage and adoption data
gap_ready = itu_wide[
    itu_wide["LTE_Coverage_Pct"].notna()
    & itu_wide["Internet_Users_Pct"].notna()
]

print("\nCountry-years ready for gap analysis:",
      len(gap_ready))

print("Countries with overlapping data:",
      gap_ready["AreaCode"].nunique())

print("Overlapping years:",
      gap_ready["Year"].min(),
      "to",
      gap_ready["Year"].max())

Countries/areas: 210

Available values by indicator:
Internet_Users_Pct          5005
LTE_Coverage_Pct            2410
Mobile_Broadband_per_100    3354
dtype: int64

Country-years ready for gap analysis: 2215
Countries with overlapping data: 205
Overlapping years: 2012 to 2025


## Merging Main Broadband Dataset

In [66]:
# Merge fixed broadband into the main ITU dataset
experiment_b_df = itu_wide.merge(
    fixed_bb_clean,
    on=["AreaCode", "Year"],
    how="left",
    validate="one_to_one"
)

print("Merged dataset shape:", experiment_b_df.shape)

print("\nFixed broadband values available:",
      experiment_b_df["Fixed_Broadband_per_100"].notna().sum())

print("Duplicate country-year rows:",
      experiment_b_df.duplicated(
          subset=["AreaCode", "Year"]
      ).sum())

display(experiment_b_df.head())

Merged dataset shape: (5234, 7)

Fixed broadband values available: 4422
Duplicate country-year rows: 0


,AreaCode,AreaName,Year,LTE_Coverage_Pct,Mobile_Broadband_per_100,Internet_Users_Pct,Fixed_Broadband_per_100
0,AFG,Afghanistan,2001,NaN,NaN,0.004723,NaN
1,AFG,Afghanistan,2002,NaN,NaN,0.004561,NaN
2,AFG,Afghanistan,2003,NaN,NaN,0.087891,NaN
3,AFG,Afghanistan,2004,NaN,NaN,0.105809,0.000849
4,AFG,Afghanistan,2005,NaN,NaN,1.224148,0.000901


## Importing Electricity data

In [75]:
# Import World Bank electricity access data

electricity_df = pd.read_csv("../data/raw/All_Electricity_Data.csv",    skiprows=4)

print("Electricity access dataset shape:", electricity_df.shape)
print("Columns:", electricity_df.columns.tolist())

display(electricity_df.head())

Electricity access dataset shape: (266, 71)
Columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.000000,100.000000,100.000000,100.000000,100.000000,99.900000,100.000000,100.000000,NaN,NaN
1,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,40.223744,43.035073,44.390861,46.282371,48.127211,48.742043,50.667516,52.598171,NaN,NaN
2,Afghanistan,AFG,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,97.700000,93.400000,97.700000,97.700000,97.700000,85.300000,85.300000,87.800000,NaN,NaN
3,Africa Western and Central,AFW,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,48.902228,51.338199,51.289374,51.851866,54.366012,55.585594,57.064737,58.822166,NaN,NaN
4,Angola,AGO,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,42.900000,45.300000,45.600000,47.000000,48.200000,48.500000,51.100000,55.500000,NaN,NaN


In [76]:
# Reshape World Bank electricity data from wide to long format

# Select years relevant to Experiment B
year_columns = [str(year) for year in range(2000, 2026)]

electricity_long = electricity_df.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_columns,
    var_name="Year",
    value_name="Electricity_Access_Pct"
)

# Convert Year into an integer
electricity_long["Year"] = electricity_long["Year"].astype(int)

print("Reshaped dataset:", electricity_long.shape)
print("Countries/areas:",
      electricity_long["Country Code"].nunique())

print("Available electricity observations:",
      electricity_long["Electricity_Access_Pct"].notna().sum())

print("Duplicate country-year rows:",
      electricity_long.duplicated(
          subset=["Country Code", "Year"]
      ).sum())

display(electricity_long.head())

Reshaped dataset: (6916, 4)
Countries/areas: 266
Available electricity observations: 6550
Duplicate country-year rows: 0


,Country Name,Country Code,Year,Electricity_Access_Pct
0,Aruba,ABW,2000,91.700000
1,Africa Eastern and Southern,AFE,2000,19.917256
2,Afghanistan,AFG,2000,4.400000
3,Africa Western and Central,AFW,2000,34.268837
4,Angola,AGO,2000,24.200000


In [77]:
# Merge World Bank electricity access into Experiment B

experiment_b_df = experiment_b_df.merge(
    electricity_long[
        ["Country Code", "Year", "Electricity_Access_Pct"]
    ],
    left_on=["AreaCode", "Year"],
    right_on=["Country Code", "Year"],
    how="left",
    validate="one_to_one"
)

# Remove the redundant World Bank country code
experiment_b_df = experiment_b_df.drop(columns=["Country Code"])

print("Merged dataset shape:", experiment_b_df.shape)

print("Electricity values available:",
      experiment_b_df["Electricity_Access_Pct"].notna().sum())

print("Duplicate country-year rows:",
      experiment_b_df.duplicated(
          subset=["AreaCode", "Year"]
      ).sum())

display(experiment_b_df.head())

Merged dataset shape: (5234, 8)
Electricity values available: 4830
Duplicate country-year rows: 0


,AreaCode,AreaName,Year,LTE_Coverage_Pct,Mobile_Broadband_per_100,Internet_Users_Pct,Fixed_Broadband_per_100,Electricity_Access_Pct
0,AFG,Afghanistan,2001,NaN,NaN,0.004723,NaN,9.3
1,AFG,Afghanistan,2002,NaN,NaN,0.004561,NaN,14.1
2,AFG,Afghanistan,2003,NaN,NaN,0.087891,NaN,19.0
3,AFG,Afghanistan,2004,NaN,NaN,0.105809,0.000849,23.8
4,AFG,Afghanistan,2005,NaN,NaN,1.224148,0.000901,28.7


In [78]:
# Check missing values across the five core indicators

indicators = [
    "Internet_Users_Pct",
    "LTE_Coverage_Pct",
    "Mobile_Broadband_per_100",
    "Fixed_Broadband_per_100",
    "Electricity_Access_Pct"
]

missing_summary = pd.DataFrame({
    "Available": experiment_b_df[indicators].notna().sum(),
    "Missing": experiment_b_df[indicators].isna().sum(),
    "Available_Pct": (
        experiment_b_df[indicators].notna().mean() * 100
    ).round(2)
})

display(missing_summary)

,Available,Missing,Available_Pct
Internet_Users_Pct,5005,229,95.62
LTE_Coverage_Pct,2410,2824,46.05
Mobile_Broadband_per_100,3354,1880,64.08
Fixed_Broadband_per_100,4422,812,84.49
Electricity_Access_Pct,4830,404,92.28


## Calculating Coverage Gap Analysis

In [79]:
# Create dataset for coverage-use gap analysis

gap_df = experiment_b_df.dropna(
    subset=["LTE_Coverage_Pct", "Internet_Users_Pct"]
).copy()

# Calculate the difference in percentage points
gap_df["Coverage_Use_Gap"] = (
    gap_df["LTE_Coverage_Pct"]
    - gap_df["Internet_Users_Pct"]
)

print("Country-year observations:", len(gap_df))
print("Countries/areas:", gap_df["AreaCode"].nunique())
print("Years:", gap_df["Year"].min(), "to", gap_df["Year"].max())

print("\nGap summary:")
print(gap_df["Coverage_Use_Gap"].describe().round(2))

Country-year observations: 2215
Countries/areas: 205
Years: 2012 to 2025

Gap summary:
count    2215.00
mean        7.18
std        23.12
min       -95.21
25%        -1.48
50%         8.62
75%        20.40
max        77.54
Name: Coverage_Use_Gap, dtype: float64


In [80]:
# Investigate the largest positive and negative coverage-use gaps

columns_to_show = [
    "AreaName",
    "Year",
    "LTE_Coverage_Pct",
    "Internet_Users_Pct",
    "Coverage_Use_Gap"
]

print("Largest positive gaps:")
display(
    gap_df.nlargest(5, "Coverage_Use_Gap")[columns_to_show]
    .round(2)
)

print("\nLargest negative gaps:")
display(
    gap_df.nsmallest(5, "Coverage_Use_Gap")[columns_to_show]
    .round(2)
)

print("\nInvalid coverage percentages:",
      (~gap_df["LTE_Coverage_Pct"].between(0, 100)).sum())

print("Invalid adoption percentages:",
      (~gap_df["Internet_Users_Pct"].between(0, 100)).sum())

Largest positive gaps:


,AreaName,Year,LTE_Coverage_Pct,Internet_Users_Pct,Coverage_Use_Gap
4030,Rwanda,2019,98.90,21.36,77.54
4029,Rwanda,2018,96.60,19.17,77.43
4031,Rwanda,2020,98.07,23.80,74.27
1091,Comoros,2017,82.48,8.48,74.00
2156,India,2018,94.00,20.08,73.92



Largest negative gaps:


,AreaName,Year,LTE_Coverage_Pct,Internet_Users_Pct,Coverage_Use_Gap
2785,Liechtenstein,2014,0.0,95.21,-95.21
1216,Czech Republic,2013,0.0,74.11,-74.11
1215,Czech Republic,2012,0.0,73.43,-73.43
706,Barbados,2015,0.0,72.05,-72.05
705,Barbados,2014,0.0,71.92,-71.92



Invalid coverage percentages: 0
Invalid adoption percentages: 0


## Investigating Zero LTE Coverage Gaps

In [81]:
# Investigate zero LTE coverage observations

zero_lte = gap_df[
    gap_df["LTE_Coverage_Pct"] == 0
].copy()

print("Zero LTE observations:", len(zero_lte))

print("Countries with zero LTE:",
      zero_lte["AreaCode"].nunique())

print("\nZero LTE observations by year:")
print(
    zero_lte["Year"]
    .value_counts()
    .sort_index()
)

print("\nInternet adoption where LTE is zero:")
print(
    zero_lte["Internet_Users_Pct"]
    .describe()
    .round(2)
)

Zero LTE observations: 170
Countries with zero LTE: 70

Zero LTE observations by year:
Year
2012     5
2013     6
2014    41
2015    53
2016    30
2017    20
2018     7
2019     4
2020     2
2021     1
2022     1
Name: count, dtype: int64

Internet adoption where LTE is zero:
count    170.00
mean      30.25
std       21.48
min        0.99
25%       10.25
50%       26.52
75%       46.43
max       95.21
Name: Internet_Users_Pct, dtype: float64


In [82]:
# Examine zero LTE coverage as a share of reported LTE observations

lte_yearly = (
    gap_df.groupby("Year")
    .agg(
        LTE_Observations=("LTE_Coverage_Pct", "count"),
        Zero_LTE=("LTE_Coverage_Pct", lambda x: (x == 0).sum())
    )
    .reset_index()
)

lte_yearly["Zero_LTE_Pct"] = (
    lte_yearly["Zero_LTE"]
    / lte_yearly["LTE_Observations"] * 100
).round(2)

display(lte_yearly)

,Year,LTE_Observations,Zero_LTE,Zero_LTE_Pct
0,2012,58,5,8.62
1,2013,79,6,7.59
2,2014,150,41,27.33
3,2015,199,53,26.63
4,2016,196,30,15.31
5,2017,192,20,10.42
6,2018,164,7,4.27
7,2019,190,4,2.11
8,2020,189,2,1.06
9,2021,191,1,0.52
